# Introduction to Machine Learning

**CS156 - Finding Patterns in Data with Machine Learning**

This notebook organizes the main ideas and implementations from the Session 1 PCW:

- thinking about machine learning through **representation, probability, optimization, and generalization**,
- exploring structured data with the Iris dataset,
- filtering and visualizing data,
- understanding image data as vectors with MNIST,
- revisiting basic probability and Bayes' rule.

The goal is not only to reproduce the PCW, but to keep the concepts and code in a form that can be revisited later.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml, load_iris

## 1. A conceptual map of machine learning

A useful way to organize introductory machine learning is around four connected questions.

### Representation

**How should the information in a problem be represented?**

Depending on the problem, an observation might be represented as a:

- scalar,
- vector,
- matrix,
- tensor,
- sequence,
- graph,
- probability distribution.

A model can only use information that is encoded in its representation.

### Probability

**How should beliefs change after observing data?**

Probability gives us a language for uncertainty. Given observations, we can reason about which parameters or hypotheses are more plausible.

### Optimization

**How do we choose the model parameters that fit the data best according to some objective?**

Many learning problems can be written schematically as

$$
\theta^* = \arg\min_\theta L(\theta),
$$

where $L(\theta)$ is a loss function.

### Generalization

**Will the pattern learned from observed data continue to work on unseen data?**

A model should capture a useful relationship rather than simply memorize the training observations.

These four ideas recur throughout CS156: representation determines what a model can see, probability describes uncertainty, optimization selects parameters, and generalization asks whether what was learned transfers beyond the training sample.

## 2. Iris: tabular data as numerical features

The Iris dataset contains measurements from 150 flowers belonging to three species:

- setosa,
- versicolor,
- virginica.

Each row is an observation and each measurement is a feature. The measurements are in centimeters.

In [ ]:
iris = load_iris(as_frame=True)
iris_df = iris.frame.copy()

iris_df["species"] = iris_df["target"].map(
    dict(enumerate(iris.target_names))
)

iris_df.head()

### Filtering observations

The PCW asks us to remove flowers with sepal length greater than 5 cm. This is a simple example of selecting observations using a condition on one feature.

In [ ]:
iris_filtered = iris_df[
    iris_df["sepal length (cm)"] <= 5
].copy()

print("Original number of flowers:", len(iris_df))
print("Number after filtering:", len(iris_filtered))

iris_filtered.head()

### Visualizing two dimensions

A scatter plot uses two numerical features as coordinates. Each point is one flower.

Coloring by species lets us see whether the labels form visible clusters in this particular two-dimensional projection.

In [ ]:
for species in iris_filtered["species"].unique():
    species_data = iris_filtered[iris_filtered["species"] == species]

    plt.scatter(
        species_data["sepal length (cm)"],
        species_data["petal length (cm)"],
        label=species,
    )

plt.xlabel("Sepal length (cm)")
plt.ylabel("Petal length (cm)")
plt.title("Petal length vs. sepal length for filtered Iris data")
plt.legend(title="Species")
plt.show()

### What the plot is doing geometrically

The full Iris dataset has four measured features, so each flower can be represented as a point in a **4-dimensional feature space**:

$$
x =
\begin{bmatrix}
\text{sepal length}\\
\text{sepal width}\\
\text{petal length}\\
\text{petal width}
\end{bmatrix}.
$$

The scatter plot displays only two coordinates. It is therefore a 2D view of a higher-dimensional representation.

## 3. MNIST: images as vectors

MNIST consists of handwritten digits stored as 28 × 28 grayscale images.

A machine-learning model does not need to store the image as a conventional picture. The 28 × 28 grid can be flattened into:

\[
28 \times 28 = 784
\]

pixel features.

So one digit can be represented as a point in a 784-dimensional feature space.

In [ ]:
# The first download may take some time because sklearn retrieves MNIST from OpenML.
X_mnist, y_mnist = fetch_openml(
    "mnist_784",
    version=1,
    as_frame=True,
    return_X_y=True,
)

mnist_df = X_mnist.copy()
mnist_df["digit"] = y_mnist.astype(int)

mnist_filtered = mnist_df[
    mnist_df["digit"].isin([3, 8])
].copy()

mnist_filtered["digit"].value_counts()

### Reconstructing a digit

The 784 pixel values are stored in one long row. Reshaping them to 28 × 28 recovers the spatial grid used by the image.

In [ ]:
first_digit = mnist_filtered.iloc[0]

pixels = first_digit.drop("digit").to_numpy(dtype=float)
image = pixels.reshape(28, 28)

plt.imshow(image, cmap="gray")
plt.title(f"Digit: {int(first_digit['digit'])}")
plt.axis("off")
plt.show()

### Variation within the same class

Even two images with the same label can have different curves, thickness, position, and proportions. A useful classifier must learn a pattern that tolerates this within-class variation.

In [ ]:
examples = mnist_filtered.sample(12, random_state=42)

fig, axes = plt.subplots(3, 4, figsize=(8, 6))

for ax, (_, row) in zip(axes.flat, examples.iterrows()):
    pixels = row.drop("digit").to_numpy(dtype=float)
    ax.imshow(pixels.reshape(28, 28), cmap="gray")
    ax.set_title(f"Digit: {int(row['digit'])}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Probability review

The PCW also revisits probability because probabilistic reasoning will appear repeatedly in machine learning.

### Chance guessing: matching five objects

Suppose five watches are randomly matched to five people. What is the probability that **at least one** person receives the correct watch?

There are

$$
5! = 120
$$

possible assignments.

The number of permutations with no correct matches is the number of derangements of five objects:

$$
!5 = 44.
$$

Therefore,

$$
P(\text{at least one correct})
= 1 - \frac{44}{120}
= \frac{19}{30}
\approx 0.6333.
$$

This is an example where counting the complement is easier than counting all successful cases directly.

In [ ]:
total_assignments = math.factorial(5)
derangements_5 = 44

prob_at_least_one_correct = 1 - derangements_5 / total_assignments
prob_at_least_one_correct

### Bayes' rule with two urns

Suppose:

- Urn A contains 12 red and 3 green balls.
- Urn B contains 6 red and 9 green balls.
- The urn is chosen with equal probability.

Then

$$
P(R\mid A)=\frac{12}{15}=0.8
$$

and

$$
P(R\mid B)=\frac{6}{15}=0.4.
$$

If we observe a red ball, Bayes' rule updates our belief about which urn was selected:

$$
P(A\mid R)
=
\frac{P(R\mid A)P(A)}
{P(R\mid A)P(A)+P(R\mid B)P(B)}.
$$

With equal priors,

$$
P(A\mid R)
=
\frac{0.8(0.5)}
{0.8(0.5)+0.4(0.5)}
=
\frac{2}{3}.
$$

In [ ]:
p_a = 0.5
p_b = 0.5

p_red_given_a = 12 / 15
p_red_given_b = 6 / 15

p_a_given_red = (
    p_red_given_a * p_a
    / (p_red_given_a * p_a + p_red_given_b * p_b)
)

p_a_given_red

## 5. Main takeaways

1. **Representation matters.** Iris rows are short feature vectors; MNIST images can be represented as 784-dimensional vectors.
2. **Visualizations are projections.** A 2D plot usually shows only part of the full feature space.
3. **Probability updates beliefs from evidence.** Bayes' rule formalizes this update.
4. **Optimization chooses parameters.** Later models will learn parameters by minimizing a loss.
5. **Generalization is the real goal.** A useful model must work on observations it did not simply memorize.